In [10]:
from google.colab import drive
drive.mount('/content/drive')

!pip install duckdb xgboost scikit-learn pyarrow joblib -q

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import duckdb
import pandas as pd
import numpy as np
import joblib
import os

parquet_path = "/content/drive/MyDrive/GENIUS/IndividualsAndHouseholdsProgramValidRegistrationsV2.parquet"

con = duckdb.connect()

print("File exists:", os.path.exists(parquet_path))
print("File size GB:", os.path.getsize(parquet_path) / (1024**3))

File exists: True
File size GB: 0.4863423518836498


In [12]:
schema_df = con.execute(f"""
DESCRIBE SELECT * FROM read_parquet('{parquet_path}')
""").df()

schema_df

,column_name,column_type,null,key,default,extra
0,incidentTypeCode,VARCHAR,YES,None,None,None
1,declarationDate,DATE,YES,None,None,None
2,disasterNumber,SMALLINT,YES,None,None,None
3,county,VARCHAR,YES,None,None,None
4,fips,VARCHAR,YES,None,None,None
...,...,...,...,...,...,...
95,verifiedOwnership,BOOLEAN,YES,None,None,None
96,verifiedOccupancy,BOOLEAN,YES,None,None,None
97,appliedDate,DATE,YES,None,None,None
98,lastRefresh,TIMESTAMP WITH TIME ZONE,YES,None,None,None


In [13]:
query = f"""
select
  incidentTypeCode,
  damagedStateAbbreviation,
  county,
  damagedZipCode,
  ownRent,
  grossIncome,
  householdComposition,
  primaryResidence,
  residenceType,
  homeOwnersInsurance,
  floodInsurance,
  homeDamage,
  autoDamage,
  emergencyNeeds,
  foodNeed,
  shelterNeed,
  accessFunctionalNeeds,
  floodDamage,
  waterLevel,
  foundationDamage,
  roofDamage,
  reportedDamage,
  ihpEligible,
  ihpAmount
from read_parquet('{parquet_path}')
where ihpAmount is not null
  and damagedZipCode is not null
  and damagedStateAbbreviation is not null
using sample 1000000 rows
"""

df = con.execute(query).df()

print(df.shape)
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(999992, 24)


,incidentTypeCode,damagedStateAbbreviation,county,damagedZipCode,ownRent,grossIncome,householdComposition,primaryResidence,residenceType,homeOwnersInsurance,...,foodNeed,shelterNeed,accessFunctionalNeeds,floodDamage,waterLevel,foundationDamage,roofDamage,reportedDamage,ihpEligible,ihpAmount
0,H,MD,Baltimore (County),21222,R,"$30,001-$60,000",3,True,H,False,...,True,<NA>,False,False,16,False,False,True,True,1688.00
1,W,AR,Columbia (County),71753,O,"<$15,000",1,True,M,False,...,<NA>,<NA>,False,False,0,False,True,True,True,915.70
2,F,TX,Harris (County),77065,O,">$175,000",4,True,H,True,...,<NA>,True,False,True,10,False,False,True,True,2867.18
3,W,OH,Cuyahoga (County),44108,O,"$15,000-$30,000",4,True,H,True,...,<NA>,<NA>,False,False,4,False,False,True,True,3897.31
4,H,FL,Lee (County),33901,O,0,1,True,H,False,...,True,<NA>,False,False,0,False,False,True,True,1000.00


In [14]:
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

print("\nihpEligible value counts:")
print(df["ihpEligible"].value_counts(dropna=False))

print("\nihpAmount summary:")
print(df["ihpAmount"].describe())

print("\nZero amount rate:")
print((df["ihpAmount"] == 0).mean())

Rows: 999992
Columns: ['incidentTypeCode', 'damagedStateAbbreviation', 'county', 'damagedZipCode', 'ownRent', 'grossIncome', 'householdComposition', 'primaryResidence', 'residenceType', 'homeOwnersInsurance', 'floodInsurance', 'homeDamage', 'autoDamage', 'emergencyNeeds', 'foodNeed', 'shelterNeed', 'accessFunctionalNeeds', 'floodDamage', 'waterLevel', 'foundationDamage', 'roofDamage', 'reportedDamage', 'ihpEligible', 'ihpAmount']

ihpEligible value counts:
ihpEligible
True     531676
False    468316
Name: count, dtype: int64

ihpAmount summary:
count    999992.000000
mean       1259.552897
std        2849.799548
min           0.000000
25%           0.000000
50%         223.720000
75%        1236.900000
max       96272.000000
Name: ihpAmount, dtype: float64

Zero amount rate:
0.4683197465579725


In [15]:
features = [
    "incidentTypeCode",
    "damagedStateAbbreviation",
    "county",
    "damagedZipCode",
    "ownRent",
    "grossIncome",
    "householdComposition",
    "primaryResidence",
    "residenceType",
    "homeOwnersInsurance",
    "floodInsurance",
    "homeDamage",
    "autoDamage",
    "emergencyNeeds",
    "foodNeed",
    "shelterNeed",
    "accessFunctionalNeeds",
    "floodDamage",
    "waterLevel",
    "foundationDamage",
    "roofDamage",
    "reportedDamage",
]

In [16]:
def clean_features(dataframe, feature_cols):
    X = dataframe[feature_cols].copy()

    # pandas nullable NA를 sklearn이 처리 가능한 np.nan으로 통일
    X = X.replace({pd.NA: np.nan})

    # object / string / boolean 계열은 전부 문자열 categorical로 처리
    for col in X.columns:
        dtype_str = str(X[col].dtype)

        if dtype_str in ["object", "string", "boolean", "bool"] or X[col].dtype == "object":
            X[col] = X[col].astype("string").fillna("Unknown").astype(str)

    # 나머지 컬럼은 숫자로 처리
    categorical_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()

    for col in X.columns:
        if col not in categorical_cols:
            X[col] = pd.to_numeric(X[col], errors="coerce")

    return X

In [17]:
X = clean_features(df, features)

y_cls = (
    df["ihpEligible"]
    .replace({pd.NA: False})
    .fillna(False)
    .astype(int)
)

categorical_features = X.select_dtypes(include=["object", "string"]).columns.tolist()
numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

print("X shape:", X.shape)
print("Positive eligibility rate:", y_cls.mean())

print("\nCategorical features:")
print(categorical_features)

print("\nNumeric features:")
print(numeric_features)

X shape: (999992, 22)
Positive eligibility rate: 0.5316802534420275

Categorical features:
['incidentTypeCode', 'damagedStateAbbreviation', 'county', 'damagedZipCode', 'ownRent', 'grossIncome', 'householdComposition', 'primaryResidence', 'residenceType', 'homeOwnersInsurance', 'floodInsurance', 'homeDamage', 'autoDamage', 'emergencyNeeds', 'foodNeed', 'shelterNeed', 'accessFunctionalNeeds', 'floodDamage', 'foundationDamage', 'roofDamage', 'reportedDamage']

Numeric features:
['waterLevel']


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, accuracy_score
from xgboost import XGBClassifier, XGBRegressor

In [19]:
preprocess = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                max_categories=50
            ),
            categorical_features
        ),
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            numeric_features
        ),
    ]
)

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_cls,
    test_size=0.2,
    random_state=42,
    stratify=y_cls
)

clf = Pipeline([
    ("preprocess", preprocess),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        tree_method="hist",
        random_state=42
    ))
])

clf.fit(X_train, y_train)

pred_proba = clf.predict_proba(X_test)[:, 1]
pred_label = (pred_proba >= 0.5).astype(int)

auc = roc_auc_score(y_test, pred_proba)
acc = accuracy_score(y_test, pred_label)

print("Eligibility model AUC:", auc)
print("Eligibility model accuracy:", acc)

Eligibility model AUC: 0.8466267165384787
Eligibility model accuracy: 0.7598487992439962


In [21]:
df_pos = df[df["ihpAmount"] > 0].copy()

print("Positive amount rows:", len(df_pos))
print(df_pos["ihpAmount"].describe())

Positive amount rows: 531676
count    531676.000000
mean       2369.004471
std        3556.206253
min           0.610000
25%         688.660000
50%        1147.270000
75%        2566.882500
max       96272.000000
Name: ihpAmount, dtype: float64


In [22]:
X_reg = clean_features(df_pos, features)
y_reg = np.log1p(df_pos["ihpAmount"].astype(float))

categorical_features_reg = X_reg.select_dtypes(include=["object", "string"]).columns.tolist()
numeric_features_reg = X_reg.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

print("X_reg shape:", X_reg.shape)
print("y_reg shape:", y_reg.shape)

print("\nCategorical features:")
print(categorical_features_reg)

print("\nNumeric features:")
print(numeric_features_reg)

X_reg shape: (531676, 22)
y_reg shape: (531676,)

Categorical features:
['incidentTypeCode', 'damagedStateAbbreviation', 'county', 'damagedZipCode', 'ownRent', 'grossIncome', 'householdComposition', 'primaryResidence', 'residenceType', 'homeOwnersInsurance', 'floodInsurance', 'homeDamage', 'autoDamage', 'emergencyNeeds', 'foodNeed', 'shelterNeed', 'accessFunctionalNeeds', 'floodDamage', 'foundationDamage', 'roofDamage', 'reportedDamage']

Numeric features:
['waterLevel']


In [23]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

preprocess_reg = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                max_categories=50
            ),
            categorical_features_reg
        ),
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            numeric_features_reg
        ),
    ]
)

In [24]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

reg = Pipeline([
    ("preprocess", preprocess_reg),
    ("model", XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        tree_method="hist",
        random_state=42
    ))
])

reg.fit(X_train_reg, y_train_reg)

pred_log = reg.predict(X_test_reg)

pred_amount = np.expm1(pred_log)
true_amount = np.expm1(y_test_reg)

mae = mean_absolute_error(true_amount, pred_amount)
rmse = np.sqrt(mean_squared_error(true_amount, pred_amount))
r2 = r2_score(true_amount, pred_amount)

print("Amount model MAE:", mae)
print("Amount model RMSE:", rmse)
print("Amount model R2:", r2)

Amount model MAE: 1440.080892566873
Amount model RMSE: 3132.34612094413
Amount model R2: 0.2244658605370281


In [25]:
def predict_assistance(input_dict):
    input_df = pd.DataFrame([input_dict])

    # 누락된 feature는 Unknown 또는 np.nan으로 채우기
    for col in features:
        if col not in input_df.columns:
            input_df[col] = np.nan

    input_df = input_df[features]
    input_clean = clean_features(input_df, features)

    eligibility_probability = float(clf.predict_proba(input_clean)[0, 1])

    predicted_log_amount = float(reg.predict(input_clean)[0])
    predicted_amount_if_eligible = float(np.expm1(predicted_log_amount))

    low = max(0, predicted_amount_if_eligible * 0.7)
    high = predicted_amount_if_eligible * 1.3

    return {
        "eligibility_probability": eligibility_probability,
        "predicted_amount_if_eligible": predicted_amount_if_eligible,
        "estimated_low": low,
        "estimated_high": high,
    }

In [26]:
sample_input = {
    "incidentTypeCode": "H",
    "damagedStateAbbreviation": "FL",
    "county": "MIAMI-DADE",
    "damagedZipCode": "33101",
    "ownRent": "O",
    "grossIncome": "$30,001-$60,000",
    "householdComposition": "3",
    "primaryResidence": True,
    "residenceType": "H",
    "homeOwnersInsurance": False,
    "floodInsurance": False,
    "homeDamage": True,
    "autoDamage": False,
    "emergencyNeeds": True,
    "foodNeed": True,
    "shelterNeed": False,
    "accessFunctionalNeeds": False,
    "floodDamage": True,
    "waterLevel": 12,
    "foundationDamage": False,
    "roofDamage": True,
    "reportedDamage": True,
}

result = predict_assistance(sample_input)
result

{'eligibility_probability': 0.9774188995361328,
 'predicted_amount_if_eligible': 6628.33740628395,
 'estimated_low': 4639.836184398765,
 'estimated_high': 8616.838628169136}

In [27]:
model_dir = "/content/drive/MyDrive/GENIUS/fema_models"
os.makedirs(model_dir, exist_ok=True)

joblib.dump(clf, f"{model_dir}/fema_ihp_eligibility_classifier.joblib")
joblib.dump(reg, f"{model_dir}/fema_ihp_amount_regressor.joblib")

print("Saved models to:", model_dir)

Saved models to: /content/drive/MyDrive/GENIUS/fema_models


In [28]:
loaded_clf = joblib.load(f"{model_dir}/fema_ihp_eligibility_classifier.joblib")
loaded_reg = joblib.load(f"{model_dir}/fema_ihp_amount_regressor.joblib")

clf = loaded_clf
reg = loaded_reg

predict_assistance(sample_input)

{'eligibility_probability': 0.9774188995361328,
 'predicted_amount_if_eligible': 6628.33740628395,
 'estimated_low': 4639.836184398765,
 'estimated_high': 8616.838628169136}

In [30]:
print("Eligibility AUC:", auc)
print("Eligibility Accuracy:", acc)
print("Amount MAE:", mae)
print("Amount RMSE:", rmse)
print("Amount R2:", r2)
print("Training rows:", len(df))
print("Positive amount rows:", len(df_pos))

Eligibility AUC: 0.8466267165384787
Eligibility Accuracy: 0.7598487992439962
Amount MAE: 1440.080892566873
Amount RMSE: 3132.34612094413
Amount R2: 0.2244658605370281
Training rows: 999992
Positive amount rows: 531676


In [29]:
metrics = {
    "eligibility_auc": float(auc),
    "eligibility_accuracy": float(acc),
    "amount_mae": float(mae),
    "amount_rmse": float(rmse),
    "amount_r2": float(r2),
    "training_rows_total": int(len(df)),
    "training_rows_positive_amount": int(len(df_pos)),
}

metrics_df = pd.DataFrame([metrics])
metrics_path = f"{model_dir}/fema_model_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

metrics_df

,eligibility_auc,eligibility_accuracy,amount_mae,amount_rmse,amount_r2,training_rows_total,training_rows_positive_amount
0,0.846627,0.759849,1440.080893,3132.346121,0.224466,999992,531676
